# Install packages

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install swig mediapy
!pip install mujoco==3.1.4
!pip install torch torchrl==0.8
!pip install gymnasium[box2d,mujoco]==0.28.1

# Imports and Functions

In [ ]:
# Imports
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from ipywidgets import interact, widgets
# Change to glfw when on Mac. This might fuck up the lighting.
%env MUJOCO_GL=egl
import mujoco
import mujoco.viewer
import mediapy as media

import torch
from torch import nn
import torch.nn.functional as F
from tensordict import TensorDict
from torchrl.data import CompositeSpec, BoundedTensorSpec, UnboundedContinuousTensorSpec
from torchrl.envs.model_based import ModelBasedEnvBase
from torchrl.modules import SafeModule
from torchrl.modules import CEMPlanner
from torchrl.modules import WorldModelWrapper
from torchrl.data import TensorDictReplayBuffer

In [ ]:
class MujocoCartPoleEnv(gym.Env):
    def __init__(self, m = 2, M = 5, l = 0.5, g = 9.81, k = 100, dt = 0.02, end_on_failure = True):
        self.model = self._get_mj_model(m,M,l,g,k,dt)
        self.data = mujoco.MjData(self.model)
        self.renderer = mujoco.Renderer(self.model)
        self.truncate = 1000
        self.action_space = gym.spaces.Box(low=-1, high=1, shape=(self.model.nu,))
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(self.model.nq + self.model.nv,))
        self.end_on_failure = end_on_failure
        self.reset()

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)
        self.data.qvel = 0.0*np.random.randn(2) # velocities
        self.data.qpos = 0.1*np.random.randn(2) # positions
        self.timestep = 0
        return self.get_state(), {}

    def set_state(self, state):
        self.data.qpos = state[:self.model.nq]
        self.data.qvel = state[self.model.nq:]

    def get_state(self):
        return np.concatenate([self.data.qpos, self.data.qvel])

    def step(self, action):
        self.data.ctrl = action
        mujoco.mj_step(self.model, self.data)
        next_state = self.get_state()
        reward = self._calculate_reward()
        terminal = self._is_done()
        truncated = self.timestep >= self.truncate
        info = {}  # Additional information (optional)
        self.timestep += 1
        return (next_state, reward, terminal, truncated, info)

    def render(self):
        self.renderer.update_scene(self.data)
        return self.renderer.render()

    def _calculate_reward(self):
        # Calculate reward based on the current state (optional)
        return int(self.data.qpos[1] > - np.pi/12 and self.data.qpos[1] < np.pi/12)

    def _is_done(self):
        # Check if the episode is done based on the current state (optional)
        if self.end_on_failure:
            return (self.data.qpos[1] < - np.pi/12 or self.data.qpos[1] > np.pi/12)
        return False

    # return a mujoco model with the given physical parameters
    @staticmethod
    def _get_mj_model(m,M,l,g,k,dt):
        xml = f"""
        <mujoco model='test_cartpole'>
            <compiler inertiafromgeom='true' coordinate='local'/>

            <size nkey="1"/>

            <option timestep='{dt}' integrator="RK4" gravity='0 0 {-g}'/>

            <default>
            <joint damping='0.0' solreflimit='.08 1'/>
            <geom contype='0' friction='0. 0. 0.'/>
            </default>

            <worldbody>
            <camera name='fixed' pos='0 -2.5 0' quat='0.707 0.707 0 0'/>
            <light name="top" castshadow="false"/>
            <geom name='floor' pos='0 0 -1' size='4 4 4' type='plane' />
            <geom name='rail1' type='capsule' pos='0 .07 0' quat='0.707 0 0.707 0'
                    size='0.02 2.2' />
            <geom name='rail2' type='capsule' pos='0 -.07 0' quat='0.707 0 0.707 0'
                    size='0.02 2.2' />
            <body name='cart' pos='0 0 0'>
                <camera name='cart' pos='0 -2.5 0' quat='0.707 0.707 0 0' />
                <joint name='slider' type='slide' limited='true' pos='0 0 0'
                        axis='1 0 0' range='-2 2' />
                <geom name='cart' type='box' pos='0 0 0'
                        mass='{M}' size='0.2 0.1 0.05' rgba='0.7 0.7 0 1' />
                <site name='cart sensor' type='box' pos='0 0 0'
                        size='0.2 0.1 0.05' rgba='0.7 0.7 0 0' />
                <body name='pole' pos='0 0 0'>
                <camera name='pole'  pos='0 -2.5 0' quat='0.707 0.707 0 0' />
                <joint name='hinge' type='hinge' pos='0 0 0' axis='0 1 0'/>
                <geom name='cpole' type='capsule' fromto='0 0 0 0 0 {l}'
                        mass='0' size='0.01 {l}' rgba='0 0.7 0.7 1' />
                <geom type='sphere' size='.05' name='tip' mass='{m}' pos='.001 0 {l}'/>
                </body>
            </body>
            </worldbody>

            <actuator>
            <motor name='slide' joint='slider' gear='{k}' ctrllimited='true' ctrlrange='-1 1' />
            </actuator>

        </mujoco>
        """
        return mujoco.MjModel.from_xml_string(xml)

In [ ]:
def render_episode(policy, env):
	frames = []
	score = 0
	state = env.get_state()
	terminated = truncated = False
	while not terminated and not truncated:
		frame = env.render()
		frames.append(frame)
		action = policy(state)
		state, reward, terminated, truncated, _ = env.step(action)
		score += reward
	env.close()
	print(f"Finished episode. Cummulative Return: {score}")
	media.show_video(frames, fps=100, loop=True)

# Model-Predictive Control
Model Predictive Control (MPC) is a control strategy that utilizes a predictive model of the system dynamics to make optimal control decisions. Unlike traditional control methods like Linear Quadratic Regulator (LQR), which rely on a fixed control law, MPC considers the future behavior of the system and optimizes the control inputs over a finite time horizon. This allows MPC to handle complex systems with constraints and uncertainties more effectively. While LQR provides optimal control for linear systems with known dynamics, MPC can handle nonlinear systems and adapt to changing operating conditions. Additionally, MPC can incorporate constraints on control inputs, states, and outputs, ensuring that the system operates within safe and desired limits. By considering the future consequences of control actions, MPC enables better performance and robustness in various applications, such as robotics, process control, and autonomous vehicles.

## Forward Models
Forward dynamics models are mathematical models that describe the behavior of a system over time. In the context of robotics and control systems, forward dynamics models are used to predict the future state of a system given its current state and the applied control inputs. These models capture the relationships between the system's state variables, such as position, velocity, and acceleration, and the forces or torques acting on the system. By simulating the dynamics of a system forward in time, forward dynamics models enable us to understand and analyze the system's behavior, plan optimal control strategies, and simulate the system's response to different inputs.

In [ ]:
# Forward model that takes in a state and an action to predict the next action using the non-linear equations of motion that we derived
# in the control exercise. The model uses a simpler integration method than MuJoCo, resulting in different trajectories.
def predict_next_states(states, actions):
	m = 2
	M = 5
	l = 0.5
	g = 9.81
	k = 100
	dt = 0.02
	xs, thetas, xs_dot, thetas_dot = states.T
	actions = actions.squeeze()

	# Non-linear equations of motion for the cart-pole system
	def equations_of_motion(xs, thetas, xs_dot, thetas_dot, actions):
		sin_thetas = torch.sin(thetas)
		cos_thetas = torch.cos(thetas)
		xs_dot_dot = (k * actions + m * l * thetas_dot**2 * sin_thetas - m * g * sin_thetas * cos_thetas) / (M + m - m * cos_thetas**2)
		thetas_dot_dot = 1/l * (g * sin_thetas - cos_thetas * xs_dot_dot)
		return xs_dot, thetas_dot, xs_dot_dot, thetas_dot_dot

	# Integrate the equations of motion using simple Euler's method
	xs_dot, thetas_dot, xs_dot_dot, thetas_dot_dot = equations_of_motion(xs, thetas, xs_dot, thetas_dot, actions)
	xs_dot += xs_dot_dot * dt
	thetas_dot += thetas_dot_dot * dt
	xs += xs_dot * dt
	thetas += thetas_dot * dt

	return torch.stack([xs, thetas, xs_dot, thetas_dot]).T

In [ ]:
# We generate an example trajectory to compare the true and predicted states for 1000 ms
trajectory_true = []
trajectory_pred = []
env = MujocoCartPoleEnv(end_on_failure=False)
dt = 0.02
state = env.reset()[0]
trajectory_true.append(state)
trajectory_pred.append(state)
state = torch.tensor(state)
sim_time = 1000
for i in range(int(sim_time*dt)):
    action = np.random.uniform(-1, 1) # random input
    trajectory_true.append(env.step(action)[0])
    trajectory_pred.append(predict_next_states(torch.tensor([trajectory_pred[-1]]), torch.tensor([action])).cpu().numpy()[0])
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot([x[0] for x in trajectory_true], 'b-', label='$x_{true}$')
ax.plot([x[0] for x in trajectory_pred], 'b--', label='$x_{predicted}$')
ax.plot([x[1] for x in trajectory_true], 'r-', label='$\\theta_{true}$')
ax.plot([x[1] for x in trajectory_pred], 'r--', label='$\\theta_{predicted}$')
ax.set_xticks(np.linspace(0, sim_time*dt, 6))
ax.set_xticklabels(np.linspace(0, sim_time,6))
ax.set_xlabel('t[ms]')
ax.legend()

### Tasks
1. Check how well our predictive model approximates the true trajectories for short an long timedistances.
2. In the 'Introduction to Control' exercise we have linearized the same model to use in LQR. Can we use the non-linear model for LQR?
3. What are advantages of the non-linear model?

## Planning with a forward model
Once we have a forward dynamics model, there are multiple approaches to planning with the model. Planning typically involves a process of adapting a hypothetical sequence of future states and actions in order to optimize an objective function. Note that this is very similar to LQR (which we have seen in the Intro to Control exercise); however, without the constraints on the dynamics and objective function. Also, other than LQR, MPC optimizes the objective function only over a fixed number of timesteps, called the horizon. The state transitions are given by the dynamics function. To choose promising actions, multiple approaches exist.
- Gradient-based methods, optimize the objective function by iteratively updating the sequence of future states and actions. These methods leverage the gradients of the objective function with respect to the states and actions to guide the optimization process towards better solutions. Gradient-based methods can converge to high-quality solutions efficiently but may get stuck in local optima.
- Sampling-based methods, explore the state-action space by randomly sampling and evaluating different sequences of future states and actions. These methods use statistical sampling techniques to guide the search towards promising regions of the state-action space, aiming to find optimal or near-optimal solutions. Sampling-based methods can explore a wider range of solutions but may require more computational resources and time.

In [ ]:
# Objective function
def quadratic_cost(trajectory, desired_state = torch.zeros(4), weight = torch.tensor([1.,2.,0.,0.])):
	with torch.no_grad():
		return torch.square(trajectory-desired_state)@weight

In [ ]:
# predict a state trajectory given an initial state and a sequence of actions, using the given dynamics model
def predict_trajectories(states, action_sequences, dynamics_model):
    trajectories = torch.zeros((len(action_sequences), len(action_sequences[0]), len(states[0])), device=states.device)
    for i, actions in enumerate(action_sequences.T):
        states = dynamics_model(states, actions)
        trajectories[:,i,:] = states
    return trajectories

In [ ]:
# predict a trajectory and evaluate it using the given objective function
def evaluate_action_sequences(state, action_sequences, desired_state=torch.zeros(4), dynamics_model=predict_next_states, objective_function=quadratic_cost, cost_weights = torch.tensor([1.,2.,0.,0.])):
    trajectories= predict_trajectories(torch.tile(state,(len(action_sequences),1)), action_sequences, dynamics_model)
    return objective_function(trajectories, desired_state, cost_weights).sum(dim=1)

In [ ]:
## Main
# Sampling-Based Planner
def plan(state, desired_state, dynamics_model, objective_function, cost_weights, horizon = 50, num_candidates=20000):
	### Your code here ###

In [ ]:
# Simulate the cart-pole system with MPC
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
desired_state = torch.zeros(4).to(device)
cost_weights = torch.tensor([1.,2.,0.1,0.1]).to(device)
env = MujocoCartPoleEnv(end_on_failure=False)
env.reset()
render_episode(lambda s: plan(torch.tensor(s).to(device), desired_state, predict_next_states, quadratic_cost, cost_weights).cpu().numpy(), env)

### Tasks
1. Implement a very simple random shooting planning algorithm. Random shooting simply selects actions at random. We generate many hypotetical trajectories and choose the one that scores highest on the objective function. We typically use receding horizon control, which means that we replan in every decision step. You can play around a bit with the number of candidates and the horizon.
2. There exist many more sophisticated planners than random shooting. Nevertheless, it is often used in MPC. What are advantages and disadvantages.
3. "All models are wrong but some are useful". What issues do you see when iteratively applying a forward dynamics model?

# CEM Planner
The Cross-Entropy Method (CEM) planner in Model Predictive Control (MPC) is a powerful tool for handling optimization in high-dimensional, continuous spaces. It operates by maintaining a distribution over possible solutions (action sequences in the case of MPC) and iteratively refines this distribution. In each iteration, it samples action sequences from the distribution, evaluates them using the cost function, and then fits the distribution to the best performing sequences. This process is repeated until convergence. In the context of MPC, the CEM planner applies the first action of the best sequence found, and the process is repeated at each time step, making it a form of receding horizon control. The CEM planner is particularly effective when the cost function is non-differentiable or has multiple local minima.

In [ ]:
# Wrapper for the world model
class MyTorchRLModel(ModelBasedEnvBase):
    def __init__(self, world_model, device="cpu", batch_size=None):
        super().__init__(world_model, device=device, batch_size=batch_size)
        self.state_spec = CompositeSpec(
            state=UnboundedContinuousTensorSpec((4,))
        )
        self.observation_spec = CompositeSpec(
            state=UnboundedContinuousTensorSpec((4,))
        )
        self.action_spec = BoundedTensorSpec(-1,1,(1,))
        self.reward_spec = UnboundedContinuousTensorSpec((1,))

    def _reset(self, tensordict: TensorDict) -> TensorDict:
        tensordict = TensorDict(
            source={'state': torch.tensor([0.,0.,0.,0.]),
                    'action': torch.tensor([0.])},
            batch_size=self.batch_size,
            device=self.device,
        )
        return tensordict

In [ ]:
# World model
world_model = WorldModelWrapper(
    SafeModule(
        predict_next_states,
        in_keys=["state", "action"],
        out_keys=["state"],
    ),
    SafeModule(
        lambda traj: - quadratic_cost(traj),
        in_keys=["state"],
        out_keys=["reward"],
    ),
)

In [ ]:
# Policy that uses the CEM planner
def cem_policy(state, planner):
	td = TensorDict({'action': torch.tensor([0.]),
					 'done': torch.tensor([False]),
					 'state': torch.tensor(state, dtype=torch.float32),
					 'terminated': torch.tensor([False])})
	return planner(td)['action'][0]

In [ ]:
## Main
# Simulate the cart-pole system with MPC
env = MujocoCartPoleEnv(end_on_failure=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env.reset()
model = MyTorchRLModel(world_model, device=device)
planner = CEMPlanner(model, 20, 3, 1000, 3)
render_episode(lambda s: cem_policy(s,planner), env)

### Tasks
1. Play around with the parameters of the CEM planner. Find a good tradeoff of performance and computation time.
2. Play around with the initial state of the system (env.set_state(init_state)). What happens if we start out with the pole hanging down?
3. Discuss the differences of LQR and MPC.